In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: Eine Qiskit Function von Global Data Quantum


*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich für Nutzer des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) Plan verfügbar ist. Sie befinden sich im Vorschau-Release-Status und können sich ändern.
## Übersicht
Der Quantum Portfolio Optimizer ist eine Qiskit Function, die das dynamische Portfolio-Optimierungsproblem löst – ein klassisches Finanzproblem, bei dem es darum geht, periodische Investitionen auf eine Reihe von Vermögenswerten aufzuteilen, um Renditen zu maximieren und Risiken zu minimieren. Durch den Einsatz modernster Quantenoptimierungstechniken vereinfacht diese Function den Prozess so weit, dass Nutzerinnen und Nutzer ohne Kenntnisse im Quantencomputing von ihren Vorteilen bei der Suche nach optimalen Investitionstrajektorien profitieren können. Ideal für Portfoliomanager, Forscher im Bereich der quantitativen Finanzen und Einzelanleger – dieses Tool ermöglicht das Backtesting von Handelsstrategien in der Portfolio-Optimierung.
## Funktionsbeschreibung
Die Quantum Portfolio Optimizer Function verwendet den Variational Quantum Eigensolver (VQE) Algorithmus, um ein Quadratic Unconstrained Binary Optimization (QUBO) Problem zu lösen und damit dynamische Portfolio-Optimierungsprobleme zu adressieren. Nutzerinnen und Nutzer müssen lediglich die Kursdaten der Vermögenswerte bereitstellen und die Investitionsbeschränkung definieren – die Function führt dann den Quantenoptimierungsprozess durch und gibt eine Reihe optimierter Investitionstrajektorien zurück.

Der Prozess besteht aus vier Hauptphasen. Zunächst werden die Eingabedaten auf ein quantenkompatibles Problem abgebildet, das QUBO des dynamischen Portfolio-Optimierungsproblems wird konstruiert und in einen Quantenoperator (Ising-Hamiltonoperator) transformiert. Anschließend werden das Eingabeproblem und der VQE-Algorithmus für den Einsatz auf Quantenhardware angepasst. Danach wird der VQE-Algorithmus auf der Quantenhardware ausgeführt, und schließlich werden die Ergebnisse nachbearbeitet, um die optimalen Investitionstrajektorien bereitzustellen. Das System umfasst auch eine rauschbewusste ([SQD](/guides/qiskit-addons-sqd)-basierte) Nachbearbeitung, um die Ausgabequalität zu maximieren.

Diese Qiskit Function basiert auf dem [veröffentlichten Manuskript](https://arxiv.org/abs/2412.19150) von Global Data Quantum.
![Visualisierung des Arbeitsablaufs der Function](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Einstieg
Authentifiziere dich mit deinem [API-Schlüssel](https://quantum.cloud.ibm.com/) und wähle die Qiskit Function wie folgt aus. (Dieses Snippet setzt voraus, dass du bereits [dein Konto gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client) in deiner lokalen Umgebung.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Beispiel: Dynamische Portfolio-Optimierung mit sieben Vermögenswerten
Dieses Beispiel zeigt, wie die dynamische Portfolio-Optimierung (DPO) Function ausgeführt und ihre Einstellungen für optimale Leistung angepasst werden. Es enthält detaillierte Schritte zur Feinabstimmung der Parameter, um die gewünschten Ergebnisse zu erzielen.

Dieser Fall umfasst sieben Vermögenswerte, vier Zeitschritte und vier Auflösungs-Qubits, was einen Gesamtbedarf von 112 Qubits ergibt.
### 1. Lies die im Portfolio enthaltenen Vermögenswerte ein
Wenn alle Vermögenswerte des Portfolios in einem Ordner unter einem bestimmten Pfad gespeichert sind, kannst du sie mit der folgenden Funktion in ein `pandas.DataFrame` laden und in ein `dict`-Objekt konvertieren.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

In diesem Beispiel haben wir die Vermögenswerte [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) und [XS2239553048](https://finance.yahoo.com/quote/XS2239553048) verwendet. Die folgende Abbildung veranschaulicht die in diesem Beispiel verwendeten Daten und zeigt die Entwicklung der täglichen Schlusskurse der Vermögenswerte vom 1. Januar bis 1. September 2023.

In diesem Beispiel haben wir Nicht-Handelstage mit dem Schlusskurs des letzten verfügbaren Datums aufgefüllt, um eine einheitliche Datumsstruktur zu gewährleisten. Dieser Schritt ist notwendig, da die ausgewählten Vermögenswerte aus verschiedenen Märkten mit unterschiedlichen Handelstagen stammen, was eine Standardisierung des Datensatzes für die Konsistenz erforderlich macht.
![Visualisierung der historischen Daten der Vermögenswerte](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Definiere das Problem
Definiere die Spezifikationen des Problems, indem du die Parameter im `qubo_settings`-Dictionary konfigurierst.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Definiere die Optimierer- und Ansatz-Einstellungen (Optional)
Definiere optional spezifische Anforderungen für den Optimierungsprozess, einschließlich der Auswahl des Optimierers und seiner Parameter sowie der Spezifikation des Primitives und seiner Konfigurationen.

Für den Tailored Ansatz basiert die gewählte Populationsgröße auf früheren Experimenten, die zeigten, dass dieser Wert eine stabile und effiziente Optimierung liefert.

Im Fall des Real Amplitudes Ansatzes kannst du einer linearen Beziehung zwischen der ``population_size`` und der Anzahl der Qubits im Circuit folgen. Als grobe Faustregel wird empfohlen, eine **Mindest-**``population_size ~ 0.8 * n_qubits`` für den `real_amplitudes`-Ansatz zu verwenden.

Es wird erwartet, dass der Optimized Real Amplitudes Ansatz eine bessere Optimierungsleistung als der Real Amplitudes Ansatz erzielt. Jedoch steigt die Anzahl der zu optimierenden Variablen in diesem Ansatz viel schneller als beim Real Amplitudes Ansatz (siehe [Manuskript](https://arxiv.org/pdf/2412.19150)). Daher erfordert der Optimized Real Amplitudes Ansatz bei großen Problemen mehr Circuit-Ausführungen. Der Optimized Real Amplitudes Ansatz ist wahrscheinlich für Probleme mit bis zu 100 Qubits nützlich, aber es wird empfohlen, bei der Einstellung der ``population_size``-Parameter achtsam zu sein. Als Beispiel für diesen Skalierungseffekt zeigt die vorherige Tabelle, dass für ein 84-Qubit-Problem der Optimized Real Amplitudes Ansatz eine ``population_size`` von 120 benötigt, während für ein 56-Qubit-Problem eine ``population_size`` von 40 ausreicht.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

Es ist auch möglich, einen bestimmten Ansatz auszuwählen. Im Folgenden wird der ``'Tailored'``-Ansatz verwendet.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Führe das Problem aus

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Ergebnisse abrufen
Wie im Abschnitt [Ausgabe](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) der API-Referenz beschrieben, gibt die Function ein Dictionary mit den Investitionstrajektorien zurück, geordnet vom niedrigsten zum höchsten nach ihrem Zielfunktionswert. Diese Ergebnismenge ermöglicht die Identifizierung der Trajektorie mit den niedrigsten Kosten und den dazugehörigen Investitionsbewertungen. Darüber hinaus ermöglicht sie die Analyse verschiedener Trajektorien und erleichtert die Auswahl derjenigen, die am besten mit spezifischen Bedürfnissen oder Zielen übereinstimmen. Diese Flexibilität stellt sicher, dass Entscheidungen auf eine Vielzahl von Präferenzen oder Szenarien zugeschnitten werden können.
Beginne damit, die resultierende Strategie darzustellen, die die niedrigsten Zielkosten im Prozess erzielt hat.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Anschließend kannst du mithilfe der Metadaten auf die Ergebnisse aller abgetasteten Strategien zugreifen. Damit kannst du die alternativen Trajektorien, die vom Optimierer zurückgegeben wurden, weiter analysieren. Lese dazu das im Dictionary `dpo_result['metadata']['all_samples_metrics']` gespeicherte Dictionary, das nicht nur zusätzliche Informationen über die optimale Strategie enthält, sondern auch Details zu den anderen Kandidatenstrategien, die während der Optimierung ausgewertet wurden.

Das folgende Beispiel zeigt, wie diese Informationen mit `pandas` gelesen werden können, um wichtige Metriken der optimalen Strategie zu extrahieren. Dazu gehören Beschränkungsabweichung, Sharpe Ratio und die entsprechende Investitionsrendite.